# Synthetic Model Training Dry Run

This notebook tests the planned modelling pipeline using synthetic data only.

These outputs are not study results and will not be used in the final analysis.

In [6]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

In [7]:
rng = np.random.default_rng(RANDOM_STATE)

n_samples = 2000
n_mut_features = 50
n_cna_features = 50

X_mut = rng.binomial(
    1,
    0.10,
    size=(n_samples, n_mut_features)
)

X_cna = rng.binomial(
    1,
    0.15,
    size=(n_samples, n_cna_features)
)

# Add some artificial signal so models have something to learn
score = (
    1.2 * X_mut[:, 0]
    + 0.8 * X_mut[:, 1]
    + 1.5 * X_cna[:, 0]
    + 1.0 * X_cna[:, 1]
    + rng.normal(0, 1, n_samples)
)

y = (score > np.median(score)).astype(int)

print("Stage 1–3:", (y == 0).sum())
print("Stage 4:", (y == 1).sum())

Stage 1–3: 1000
Stage 4: 1000


In [8]:
X_mut = pd.DataFrame(
    X_mut,
    columns=[f"MUT_gene_{i}" for i in range(n_mut_features)]
)

X_cna = pd.DataFrame(
    X_cna,
    columns=[f"CNA_gene_{i}" for i in range(n_cna_features)]
)

X_combined = pd.concat([X_mut, X_cna], axis=1)

print("Mutations:", X_mut.shape)
print("CNA:", X_cna.shape)
print("Combined:", X_combined.shape)

Mutations: (2000, 50)
CNA: (2000, 50)
Combined: (2000, 100)


In [9]:
indices = np.arange(n_samples)

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=y[temp_idx],
    random_state=RANDOM_STATE
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Train: 1400
Validation: 300
Test: 300


In [10]:
#Define model families
def make_logistic_regression():
    return LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    )


def make_random_forest():
    return RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

In [11]:
def train_and_validate(model, X, y, train_idx, val_idx):
    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y[train_idx]
    y_val = y[val_idx]

    model.fit(X_train, y_train)

    val_prob = model.predict_proba(X_val)[:, 1]

    auroc = roc_auc_score(y_val, val_prob)

    return model, auroc

In [12]:
feature_sets = {
    "Mutations": X_mut,
    "CNA": X_cna,
    "Combined": X_combined
}

model_builders = {
    "Logistic Regression": make_logistic_regression,
    "Random Forest": make_random_forest
}

results = []
trained_models = {}

for feature_name, X in feature_sets.items():

    for model_name, build_model in model_builders.items():

        model = build_model()

        model, val_auroc = train_and_validate(
            model,
            X,
            y,
            train_idx,
            val_idx
        )

        trained_models[(feature_name, model_name)] = model

        results.append({
            "Feature Set": feature_name,
            "Model": model_name,
            "Validation AUROC": val_auroc
        })

In [13]:
results_df = pd.DataFrame(results)

results_df

,Feature Set,Model,Validation AUROC
0,Mutations,Logistic Regression,0.634222
1,Mutations,Random Forest,0.586289
2,CNA,Logistic Regression,0.660622
3,CNA,Random Forest,0.667600
4,Combined,Logistic Regression,0.739956
5,Combined,Random Forest,0.750622


## Dry-run result

All six planned model configurations trained successfully using synthetic data.

The combined feature models achieved the highest validation AUROC, as expected,
because the synthetic outcome was intentionally generated using both mutation
and CNA features.

These values are only a pipeline validation and are not project results.